# `03_memory.ipynb`

## Key Concept
- State에 `messages` 항목에 대한 설명 -> Graph 내부에서 턴마다 필요한 메세지(AI, Human, System, Tool)를 쌓는 용도
- InMemorySaver로 테스트 -> 여러 턴에 대해 저장하자
- PostgreSQL 에 직접 저장하는 방법

In [38]:
from dotenv import load_dotenv
load_dotenv()

True

In [39]:
# 직접 만들기 (교육용)
from typing import TypedDict, Annotated
from langgraph.graph import add_messages

class MyState(TypedDict):
    # messages: list  # 그냥 리스트임 -> 교체해야하면 교체됨
    messages: Annotated[list, add_messages]  # 교체할 타이밍에, 교체하지 않고 쌓아 나가는 기능을 추가해주세요
    is_good: bool
    

In [40]:
# 앞으로 모든 state는 이렇게 만든다.
from langgraph.graph import MessagesState

class MyState(MessagesState):
    # messages 기능 자동 탑재
    is_good: bool

In [41]:
# node
from langchain.chat_models import init_chat_model

llm = init_chat_model('openai:gpt-4.1-mini')

# is_good 처리
def node_a(state: MyState):
    return {'is_good': True}  # 기존 state의 'is_good' 을 바꿔주세요


# AI 답변 생성
def node_b(state: MyState):
    messages = state['messages']
    ai_msg = llm.invoke(messages)
    return {'messages': [ai_msg]}  # 기존 state의 'messages'를 교체해 주세요

In [68]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

graph = StateGraph(MyState)
graph.add_node(node_a)
graph.add_node(node_b)
graph.add_edge(START, 'node_a')
graph.add_edge('node_a', 'node_b')
graph.add_edge('node_b', END)


In [66]:
from langchain.messages import HumanMessage
# 테스트용 메모리
workflow = graph.compile(checkpointer=InMemorySaver())
config = {'configurable': {'thread_id': '123-456'}}  # 각 세션의 고유 id로 구분

result = workflow.invoke(
    {'messages': [HumanMessage('질문을 몇 번이나 했지?')]},  # 1번 인자: state
    config,  # 2번 인자, 설정값
)

In [67]:
for msg in result['messages']:
    msg.pretty_print()

================================ Human Message =================================

질문을 몇 번이나 했지?
================================== Ai Message ==================================

지금까지 1번 질문하셨습니다. 더 궁금한 점 있으면 언제든지 물어보세요!


# Postgres 영구저장

In [61]:
# 영구저장 메모리
# uv add langgraph-checkpoint-postgres psycopg[binary]
import os
from dotenv import load_dotenv
from langgraph.checkpoint.postgres import PostgresSaver

load_dotenv()
DB_URI = os.getenv('POSTGRES_URI')

with PostgresSaver.from_conn_string(DB_URI) as checkpointer:
    # 현재 비어있는 DB에 테이블 생성 및 초기화
    checkpointer.setup()
    pg_workflow = graph.compile(checkpointer=checkpointer)

    config = {'configurable': {'thread_id': 'abc-1234'}}
    init_state = {'messages': [
        HumanMessage('기억하고 있는 것 맞아?'),
    ]}
    result = pg_workflow.invoke(init_state, config)
    

In [62]:
result

{'messages': [HumanMessage(content='지금 외부 DB인 수파베이스랑 연결해놨잖아?', additional_kwargs={}, response_metadata={}, id='222c59dd-5cbb-4c43-8f1c-69bf6383b757'),
  AIMessage(content='아쉽게도 지금은 외부 DB나 특정 서비스(예: 수파베이스)와 직접 연결되어 있지 않습니다. 다만, 수파베이스 연결 방법이나 관련 코드 작성에 대해 도움을 드릴 수 있으니 필요하시면 알려주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 26, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_a6a353f320', 'id': 'chatcmpl-EJaeWi0NIWcLg4pvBgsyHKFrfKYfT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0613d-c2e2-7952-a1b1-92e503c504dc-0', to